# 12.6 计算机使用智能体 (Computer Use Agent)

> 🕐 预估学习时间：30分钟

计算机使用智能体（Computer Use Agent）是让 LLM 像人类一样操作计算机图形界面（GUI）的技术。通过视觉理解屏幕内容并生成鼠标/键盘动作，Agent 可以完成浏览器操作、表单填写、文件管理等复杂任务。

本节涵盖：
- Computer Use 概念与挑战
- 屏幕表示与元素解析
- 动作空间设计
- 视觉-动作对齐（Set-of-Mark）
- 任务规划与执行
- 安全与限制机制

## 1. Computer Use 概述

**什么是 Computer Use**：
- 让 LLM 通过视觉感知屏幕，生成鼠标/键盘动作来操作 GUI
- 输入：屏幕截图（或辅助功能树）
- 输出：结构化动作（点击坐标、输入文本、滚动等）

**代表系统**：
- **Claude Computer Use**：Anthropic 推出的计算机使用能力
- **GPT-4V + Action**：OpenAI 视觉模型结合动作生成
- **OmniParser**：微软提出的屏幕解析方法

**核心挑战**：
- 屏幕元素定位精度（像素级坐标）
- 动作空间的高维性与连续性
- 长程任务的规划与错误恢复
- 安全性（避免误操作敏感数据）

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple, Any
from enum import Enum

torch.manual_seed(42)


class ElementType(Enum):
    BUTTON = 'button'
    TEXT = 'text'
    INPUT = 'input'
    IMAGE = 'image'
    LINK = 'link'
    CONTAINER = 'container'


@dataclass
class ScreenElement:
    element_id: int
    element_type: ElementType
    bbox: Tuple[int, int, int, int]
    text: str = ''
    is_actionable: bool = False

    def center(self) -> Tuple[int, int]:
        return (self.bbox[0] + self.bbox[2] // 2, self.bbox[1] + self.bbox[3] // 2)


@dataclass
class ScreenState:
    width: int
    height: int
    elements: List[ScreenElement] = field(default_factory=list)

    def get_actionable_elements(self) -> List[ScreenElement]:
        return [e for e in self.elements if e.is_actionable]


class ScreenParser:
    def __init__(self):
        self.next_id = 0

    def parse(self, raw_elements: List[Dict]) -> ScreenState:
        elements = []
        for raw in raw_elements:
            elem = ScreenElement(
                element_id=self.next_id,
                element_type=ElementType(raw.get('type', 'text')),
                bbox=tuple(raw['bbox']),
                text=raw.get('text', ''),
                is_actionable=raw.get('actionable', False)
            )
            elements.append(elem)
            self.next_id += 1
        return ScreenState(width=1920, height=1080, elements=elements)


print('=== Screen Representation ===')
parser = ScreenParser()
raw_screen = [
    {'type': 'text', 'bbox': [100, 50, 300, 40], 'text': '用户登录', 'actionable': False},
    {'type': 'input', 'bbox': [100, 120, 200, 30], 'text': '用户名', 'actionable': True},
    {'type': 'input', 'bbox': [100, 170, 200, 30], 'text': '密码', 'actionable': True},
    {'type': 'button', 'bbox': [100, 220, 100, 40], 'text': '登录', 'actionable': True},
    {'type': 'link', 'bbox': [100, 280, 150, 20], 'text': '忘记密码', 'actionable': True},
]

screen = parser.parse(raw_screen)
print(f'Screen size: {screen.width}x{screen.height}')
print(f'Total elements: {len(screen.elements)}')
for elem in screen.elements:
    etype = elem.element_type.value
    print(f'  [{elem.element_id}] {etype}: {elem.text} at {elem.bbox}')

actionable = screen.get_actionable_elements()
print(f'\nActionable elements: {len(actionable)}')

print(f'\nKey: Screen representation abstracts GUI into structured elements with bbox and type.')
print(f'This enables the agent to reason about what is on screen and where to act.')

## 2. 动作空间设计

**动作类型**：
- **click(x, y)**：点击坐标
- **type(text)**：输入文本
- **scroll(direction, amount)**：滚动
- **drag(x1, y1, x2, y2)**：拖拽
- **keypress(key)**：按键
- **wait(duration)**：等待

**动作表示**：
- 离散动作：按键、点击类型（左/右/双击）
- 连续动作：坐标位置
- 混合空间：需要特殊处理（如分桶化坐标）

**设计考量**：
- 动作空间需覆盖常见 GUI 交互
- 坐标归一化便于跨分辨率泛化
- 动作编码可输入策略网络用于 RL 训练

In [ ]:
class ActionType(Enum):
    CLICK = 'click'
    TYPE = 'type'
    SCROLL = 'scroll'
    DRAG = 'drag'
    KEYPRESS = 'keypress'
    WAIT = 'wait'


@dataclass
class Action:
    action_type: ActionType
    parameters: Dict[str, Any] = field(default_factory=dict)

    def describe(self) -> str:
        params = ', '.join(f'{k}={v}' for k, v in self.parameters.items())
        return f'{self.action_type.value}({params})'


class ClickAction(Action):
    def __init__(self, x: int, y: int, button: str = 'left'):
        super().__init__(ActionType.CLICK, {'x': x, 'y': y, 'button': button})


class TypeAction(Action):
    def __init__(self, text: str):
        super().__init__(ActionType.TYPE, {'text': text})


class ScrollAction(Action):
    def __init__(self, direction: str, amount: int = 3):
        super().__init__(ActionType.SCROLL, {'direction': direction, 'amount': amount})


class DragAction(Action):
    def __init__(self, x1: int, y1: int, x2: int, y2: int):
        super().__init__(ActionType.DRAG, {'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2})


class ActionSpace:
    def __init__(self, screen_width: int = 1920, screen_height: int = 1080):
        self.screen_width = screen_width
        self.screen_height = screen_height
        self.action_types = list(ActionType)

    def encode(self, action: Action) -> torch.Tensor:
        type_idx = self.action_types.index(action.action_type)
        type_onehot = F.one_hot(torch.tensor(type_idx), num_classes=len(self.action_types)).float()
        x = action.parameters.get('x', 0) / self.screen_width
        y = action.parameters.get('y', 0) / self.screen_height
        coords = torch.tensor([x, y], dtype=torch.float)
        return torch.cat([type_onehot, coords])

    def num_actions(self) -> int:
        return len(self.action_types)


class ActionExecutor:
    def __init__(self, screen: ScreenState):
        self.screen = screen
        self.action_log: List[str] = []

    def execute(self, action: Action) -> Dict[str, Any]:
        result = {'success': False, 'message': ''}
        if action.action_type == ActionType.CLICK:
            x = action.parameters['x']
            y = action.parameters['y']
            hit = None
            for elem in self.screen.elements:
                ex, ey, ew, eh = elem.bbox
                if ex <= x <= ex + ew and ey <= y <= ey + eh:
                    hit = elem
                    break
            if hit:
                etype = hit.element_type.value
                result = {'success': True, 'message': f'Clicked {etype}: {hit.text}', 'target': hit.element_id}
            else:
                result = {'success': False, 'message': f'Click at ({x},{y}) hit nothing'}
        elif action.action_type == ActionType.TYPE:
            text = action.parameters['text']
            result = {'success': True, 'message': f'Typed: {text}'}
        elif action.action_type == ActionType.SCROLL:
            direction = action.parameters['direction']
            amount = action.parameters['amount']
            result = {'success': True, 'message': f'Scrolled {direction} by {amount}'}
        elif action.action_type == ActionType.DRAG:
            x1 = action.parameters['x1']
            y1 = action.parameters['y1']
            x2 = action.parameters['x2']
            y2 = action.parameters['y2']
            result = {'success': True, 'message': f'Dragged from ({x1},{y1}) to ({x2},{y2})'}

        msg = result['message']
        self.action_log.append(f'{action.describe()} -> {msg}')
        return result


print('=== Action Space Design ===')
action_space = ActionSpace()
print(f'Number of action types: {action_space.num_actions()}')

sample_action = ClickAction(150, 240)
encoded = action_space.encode(sample_action)
print(f'Encoded click action: shape={encoded.shape}, values={encoded.tolist()}')

executor = ActionExecutor(screen)
actions = [
    ClickAction(150, 135),
    TypeAction('alice'),
    ClickAction(150, 185),
    TypeAction('secret123'),
    ClickAction(150, 240),
]

print('\nExecuting action sequence:')
for i, action in enumerate(actions):
    result = executor.execute(action)
    status = 'OK' if result['success'] else 'FAIL'
    msg = result['message']
    print(f'  Step {i+1}: [{status}] {msg}')

print(f'\nKey: Action space defines what the agent can do; executor applies actions to screen state.')
print(f'Encoding actions as tensors enables learning action policies via RL or imitation.')

## 3. 视觉-动作对齐

**Set-of-Mark (SoM) Prompting**：
- 在屏幕截图上为每个可交互元素标注编号
- LLM 通过引用编号来选择目标元素
- 解决了"看到但无法精确指向"的问题

**辅助功能树（Accessibility Tree）**：
- 操作系统提供的语义化 UI 树结构
- 包含元素角色（role）、名称（name）、状态等信息
- 比纯视觉更精确，但并非所有应用都暴露

**对齐策略**：
- 视觉标记 ↔ 辅助功能树节点 ↔ 动作目标
- 多模态融合提高定位鲁棒性

In [ ]:
class SetOfMarkAnnotator:
    def __init__(self):
        self.marks: Dict[int, int] = {}

    def annotate(self, screen: ScreenState) -> List[Dict]:
        annotations = []
        mark_num = 0
        for elem in screen.elements:
            if elem.is_actionable:
                mark_num += 1
                self.marks[mark_num] = elem.element_id
                cx, cy = elem.center()
                annotations.append({
                    'mark': mark_num,
                    'element_id': elem.element_id,
                    'center': (cx, cy),
                    'text': elem.text,
                    'bbox': elem.bbox
                })
        return annotations

    def get_element_by_mark(self, mark_num: int) -> Optional[int]:
        return self.marks.get(mark_num)


class AccessibilityTreeNode:
    def __init__(self, name: str, role: str, element_id: Optional[int] = None):
        self.name = name
        self.role = role
        self.element_id = element_id
        self.children: List['AccessibilityTreeNode'] = []

    def add_child(self, child: 'AccessibilityTreeNode'):
        self.children.append(child)

    def to_dict(self) -> Dict:
        return {
            'name': self.name,
            'role': self.role,
            'element_id': self.element_id,
            'children': [c.to_dict() for c in self.children]
        }


class AccessibilityTree:
    def __init__(self):
        self.root: Optional[AccessibilityTreeNode] = None

    def build_from_screen(self, screen: ScreenState) -> AccessibilityTreeNode:
        self.root = AccessibilityTreeNode('Window', 'window')
        for elem in screen.elements:
            role = elem.element_type.value
            node = AccessibilityTreeNode(elem.text or role, role, elem.element_id)
            self.root.add_child(node)
        return self.root

    def find_actionable(self) -> List[AccessibilityTreeNode]:
        result = []
        if self.root is None:
            return result
        for child in self.root.children:
            if child.element_id is not None:
                result.append(child)
        return result


class VisualActionAligner:
    def __init__(self):
        self.annotator = SetOfMarkAnnotator()
        self.tree = AccessibilityTree()

    def align(self, screen: ScreenState) -> Dict:
        annotations = self.annotator.annotate(screen)
        self.tree.build_from_screen(screen)
        actionable_nodes = self.tree.find_actionable()
        return {
            'marks': annotations,
            'tree': self.tree.root.to_dict() if self.tree.root else {},
            'actionable_count': len(actionable_nodes)
        }

    def select_by_mark(self, mark_num: int) -> Optional[int]:
        return self.annotator.get_element_by_mark(mark_num)


print('=== Visual-Action Alignment ===')
aligner = VisualActionAligner()
alignment = aligner.align(screen)

print('Set-of-Mark annotations:')
marks = alignment['marks']
for mark in marks:
    m = mark['mark']
    text = mark['text']
    center = mark['center']
    print(f'  [{m}] {text} at center {center}')

count = alignment['actionable_count']
print(f'\nAccessibility tree actionable nodes: {count}')

print('\nSelecting elements by mark number:')
for mark_num in range(1, 5):
    elem_id = aligner.select_by_mark(mark_num)
    print(f'  Mark {mark_num} -> element_id {elem_id}')

print(f'\nKey: Set-of-Mark prompting annotates actionable elements with numbers.')
print(f'This bridges visual perception and action selection for the LLM.')

## 4. 任务规划与执行

**任务分解**：
- 将复杂任务拆分为原子动作序列
- 每步动作依赖前一步的屏幕状态
- 支持条件分支与循环

**执行循环**：
1. 观察当前屏幕状态
2. 决定下一步动作
3. 执行动作
4. 验证结果是否符合预期
5. 若失败则尝试恢复

**错误恢复**：
- 坐标偏移：重新定位元素中心
- 元素未找到：滚动或等待
- 操作超时：重试或切换策略

In [ ]:
class ComputerUseAgent:
    def __init__(self, screen: ScreenState):
        self.screen = screen
        self.executor = ActionExecutor(screen)
        self.aligner = VisualActionAligner()
        self.plan: List[Action] = []
        self.step_results: List[Dict] = []
        self.max_retries = 3

    def plan_task(self, task_description: str) -> List[Action]:
        alignment = self.aligner.align(self.screen)
        marks = alignment['marks']
        mark_by_text = {}
        for m in marks:
            mark_by_text[m['text']] = m

        plan = []
        if '登录' in task_description or 'login' in task_description.lower():
            username_mark = mark_by_text.get('用户名')
            password_mark = mark_by_text.get('密码')
            login_mark = mark_by_text.get('登录')
            if username_mark:
                cx, cy = username_mark['center']
                plan.append(ClickAction(cx, cy))
                plan.append(TypeAction('alice'))
            if password_mark:
                cx, cy = password_mark['center']
                plan.append(ClickAction(cx, cy))
                plan.append(TypeAction('p@ssw0rd'))
            if login_mark:
                cx, cy = login_mark['center']
                plan.append(ClickAction(cx, cy))
        else:
            plan.append(ClickAction(100, 100))
        self.plan = plan
        return plan

    def execute_step(self, step_idx: int) -> Dict:
        if step_idx >= len(self.plan):
            return {'success': False, 'message': 'Step out of range'}
        action = self.plan[step_idx]
        result = self.executor.execute(action)
        self.step_results.append(result)
        return result

    def recover_from_error(self, failed_step: int) -> bool:
        action = self.plan[failed_step]
        if action.action_type == ActionType.CLICK:
            x = action.parameters['x']
            y = action.parameters['y']
            for elem in self.screen.elements:
                if elem.is_actionable:
                    cx, cy = elem.center()
                    if abs(cx - x) < 50 and abs(cy - y) < 50:
                        action.parameters['x'] = cx
                        action.parameters['y'] = cy
                        return True
        return False

    def run(self, task_description: str) -> bool:
        plan = self.plan_task(task_description)
        print(f'Task plan: {len(plan)} steps')
        for i, action in enumerate(plan):
            print(f'  Step {i+1}: {action.describe()}')

        print('\nExecution:')
        for i in range(len(plan)):
            result = self.execute_step(i)
            status = 'OK' if result['success'] else 'FAIL'
            msg = result['message']
            print(f'  Step {i+1}: [{status}] {msg}')
            if not result['success']:
                print(f'  Attempting recovery...')
                if self.recover_from_error(i):
                    print(f'  Recovery: adjusted coordinates')
                    result = self.execute_step(i)
                    status = 'OK' if result['success'] else 'FAIL'
                    msg = result['message']
                    print(f'  Retry: [{status}] {msg}')
                else:
                    print(f'  Recovery failed')
                    return False
        return True


print('=== Task Planning and Execution ===')
agent = ComputerUseAgent(screen)
success = agent.run('用户登录')
print(f'\nTask completed: {success}')

print(f'\nKey: The agent decomposes tasks into action steps, executes them, and recovers from errors.')
print(f'Plan-execute-observe loop is the core of computer use agents.')

## 5. 安全与限制

**沙箱隔离（Sandboxing）**：
- 限制文件系统访问范围
- 禁用或限制网络请求
- 资源使用配额（CPU、内存、时间）

**权限管理（Permission）**：
- 动作级别权限控制（读/写/删除）
- 敏感操作需人工审批
- 基于角色的访问控制

**人机协同（Human-in-the-Loop）**：
- 高风险操作前请求用户确认
- 提供操作预览与撤销机制
- 异常情况自动暂停并通知

**速率限制**：
- 防止 Agent 失控快速执行
- 单位时间内最大动作数
- 异常检测与自动熔断

In [ ]:
class SafetyGuard:
    def __init__(self):
        self.blocked_patterns = ['rm -rf', 'format', 'del /f', 'shutdown']
        self.rate_limit = 10
        self.action_count = 0
        self.permissions: Dict[str, bool] = {
            'click': True,
            'type': True,
            'scroll': True,
            'drag': True,
            'keypress': True,
            'delete': False,
            'download': False
        }

    def check_permission(self, action_type: str) -> bool:
        return self.permissions.get(action_type, False)

    def validate_action(self, action: Action) -> Dict:
        self.action_count += 1
        if self.action_count > self.rate_limit:
            return {'allowed': False, 'reason': 'Rate limit exceeded'}

        atype = action.action_type.value
        if not self.check_permission(atype):
            return {'allowed': False, 'reason': f'Action {atype} not permitted'}

        if action.action_type == ActionType.TYPE:
            text = action.parameters.get('text', '')
            for pattern in self.blocked_patterns:
                if pattern in text.lower():
                    return {'allowed': False, 'reason': f'Blocked pattern detected: {pattern}'}

        return {'allowed': True, 'reason': 'OK'}

    def request_permission(self, action_type: str) -> bool:
        current = self.permissions.get(action_type, False)
        if not current:
            print(f'  [Permission Request] Action {action_type} requires approval')
            self.permissions[action_type] = True
            return True
        return True


class Sandbox:
    def __init__(self):
        self.file_system: Dict[str, str] = {
            '/tmp/test.txt': 'hello world',
            '/home/user/doc.md': 'sandbox content'
        }
        self.network_access = False
        self.allowed_paths = ['/tmp/']

    def execute_action(self, action: Action) -> Dict:
        if action.action_type == ActionType.TYPE:
            text = action.parameters.get('text', '')
            if text.startswith('cat '):
                path = text[4:]
                if any(path.startswith(p) for p in self.allowed_paths):
                    content = self.file_system.get(path, 'File not found')
                    return {'success': True, 'output': content}
                else:
                    return {'success': False, 'output': 'Access denied: path outside sandbox'}
            return {'success': True, 'output': f'Executed: {text}'}
        atype = action.action_type.value
        return {'success': True, 'output': f'Action {atype} executed in sandbox'}


print('=== Safety and Limitations ===')
guard = SafetyGuard()
sandbox = Sandbox()

test_actions = [
    ClickAction(100, 200),
    TypeAction('hello world'),
    TypeAction('rm -rf /'),
    ScrollAction('down', 5),
]

print('Action validation:')
for action in test_actions:
    atype = action.action_type.value
    validation = guard.validate_action(action)
    allowed = validation['allowed']
    reason = validation['reason']
    status = 'ALLOWED' if allowed else 'BLOCKED'
    print(f'  {atype}: [{status}] {reason}')

print('\nSandbox execution:')
sandbox_actions = [
    TypeAction('cat /tmp/test.txt'),
    TypeAction('cat /home/user/doc.md'),
    TypeAction('cat /etc/passwd'),
]
for action in sandbox_actions:
    text = action.parameters['text']
    result = sandbox.execute_action(action)
    success = result['success']
    output = result['output']
    status = 'OK' if success else 'DENIED'
    print(f'  {text}: [{status}] {output}')

print('\nPermission management:')
perm_before = guard.check_permission('delete')
print(f'  delete permission before: {perm_before}')
guard.request_permission('delete')
perm_after = guard.check_permission('delete')
print(f'  delete permission after: {perm_after}')

print(f'\nKey: Safety guards validate actions, sandboxes isolate execution, and permissions control capabilities.')
print(f'Human-in-the-loop approval is essential for sensitive operations.')

## 📝 课后思考题

1. Computer Use 与传统 API 工具调用有什么本质区别？各自适合什么场景？
2. Set-of-Mark prompting 如何解决视觉-动作对齐问题？有什么局限性？
3. 在生产环境中，如何平衡 Computer Use Agent 的自主性和安全性？
4. 如果要让 Agent 处理动态变化的界面（如弹窗、动画），需要哪些额外机制？